In [1]:
#default_exp identify_splice

In [2]:
#hide
from nbdev.showdoc import *


>get sequences, save as 200bp sequence


In [3]:
#export
import pybedtools
import pandas as pd

In [ ]:
#export
def get_fasta_bedtools(on_file, length_of_seq):
    #uses bedtools to fetch fasta sequences from human genome
    a = pybedtools.BedTool(on_file)
    fasta = pybedtools.example_filename('/Users/hnjacobs/Dropbox (MIT)/GradSchool/Finuance/sQTL_snakemake_wf/intersect/hg38/fasta_hg38/hg38.fa')
    a = a.sequence(fi=fasta, s=True)
    fastas=open(a.seqfn).read()

    seqs=fastas.split('\n')[1::2]
    seqs=[sequence.upper() for sequence in seqs]
    regions=fastas.split('\n')[0:-1:2]
    fastas = pd.DataFrame({'regions' : regions,
                                str(length_of_seq)+'bp_seq' : seqs})  
    return fastas

In [5]:
#export
def gc_content(seq):
    return round((seq.count('C')+seq.count('G')) / len(seq)*100,-1)
    #match for GC content of 10 bp region around variant


# maxent functions


In [3]:
#export

from SplicingStats.MaxEnt import MaxEnt
import numpy as np
def sliding_window_along_sequence(df, size):
    #sliding window along the center of the string, variant_id
    ref=df['100bp_seq']
    alt=df['alt_100bp_seq']
    
    window_ref=ref.apply(lambda x: [x[i:i+size] for i in range(100-size, 100+size+1)])
    window_alt=alt.apply(lambda x: [x[i:i+size] for i in range(100-size, 100+size+1)])
    
    df[str(size)+'bp_sequence_window']=window_ref
    df[str(size)+'bp_sequence_window_alt']=window_alt
    return df


def compute_maxent_on_data(df, ref_seqs, alt_seqs):
    
    compute_maxent=MaxEnt()
    if '23' in ref_seqs:
        ref_output=df[ref_seqs].apply(lambda x: np.max(np.array(compute_maxent.compute_score(x))))
        alt_output=df[alt_seqs].apply(lambda x: np.max(np.array(compute_maxent.compute_score(x))))
        df=df.assign(three_p_maxent_ref=ref_output)
        df=df.assign(three_p_maxent_alt=alt_output)
        
    if '9' in ref_seqs:
        ref_output=df[ref_seqs].apply(lambda x: np.max(np.array(compute_maxent.compute_score(x))))    
        alt_output=df[alt_seqs].apply(lambda x: np.max(np.array(compute_maxent.compute_score(x))))

        df=df.assign(five_p_maxent_ref=ref_output)
        df=df.assign(five_p_maxent_alt=alt_output)
    return df


find splice sites
--

In [1]:
#export
def define_ss_gencode(intersect_exons, intersect_introns):

    introns_or_exons=pd.concat([intersect_exons, intersect_introns])
    
    dist_to_Annot_Start=introns_or_exons.start-introns_or_exons.annot_start

    dist_to_Annot_End=introns_or_exons.annot_end-introns_or_exons.start

    introns_or_exons=introns_or_exons.assign(dist_to_Annot_End=dist_to_Annot_End)
    introns_or_exons=introns_or_exons.assign(dist_to_Annot_Start=dist_to_Annot_Start)

    
     #only take the closest intron or exon. 
    introns_or_exons['min_dist_to_either_end']=introns_or_exons[['dist_to_Annot_End', 'dist_to_Annot_Start']].min(axis=1)
    introns_or_exons=introns_or_exons.sort_values(by=['min_dist_to_either_end']).drop_duplicates(subset='variant_id', keep='first')




    positive_strand=introns_or_exons[introns_or_exons.strand=='+']

    negative_strand=introns_or_exons[introns_or_exons.strand=='-']

    

# + strand introns
    positive_strand_introns=positive_strand[positive_strand.region=='introns']
         
    in_range_of_5p_ss_i=positive_strand_introns.apply(lambda x:  x['dist_to_Annot_Start'] in range(0, 5), axis=1)
    in_range_of_3p_ss_i=positive_strand_introns.apply(lambda x:  x['dist_to_Annot_End'] in range(0, 19), axis=1)


    
    positive_strand_introns=positive_strand_introns.assign(in_5p_ss_gencode=in_range_of_5p_ss_i)
    positive_strand_introns=positive_strand_introns.assign(in_3p_ss_gencode=in_range_of_3p_ss_i)

# + strand exons
    positive_strand_exons=positive_strand[positive_strand.region=='exons']
         
    in_range_of_5p_ss_e=positive_strand_exons.apply(lambda x:  x['dist_to_Annot_End'] in range(0, 3), axis=1)
    in_range_of_3p_ss_e=positive_strand_exons.apply(lambda x:  x['dist_to_Annot_Start'] in range(0, 3), axis=1)


    positive_strand_exons=positive_strand_exons.assign(in_5p_ss_gencode=in_range_of_5p_ss_e)
    positive_strand_exons=positive_strand_exons.assign(in_3p_ss_gencode=in_range_of_3p_ss_e)



#####negative strand
# - strand exons
    negative_strand_exons=negative_strand[negative_strand.region=='exons']
         
    in_range_of_5p_ss_e=negative_strand_exons.apply(lambda x:  x['dist_to_Annot_Start'] in range(0, 3), axis=1)
    in_range_of_3p_ss_e=negative_strand_exons.apply(lambda x:  x['dist_to_Annot_End'] in range(0, 3), axis=1)
    
    negative_strand_exons=negative_strand_exons.assign(in_5p_ss_gencode=in_range_of_5p_ss_e)
    negative_strand_exons=negative_strand_exons.assign(in_3p_ss_gencode=in_range_of_3p_ss_e)

# - strand introns
    negative_strand_introns=negative_strand[negative_strand.region=='introns']
         
    in_range_of_5p_ss_i=negative_strand_introns.apply(lambda x:  x['dist_to_Annot_End'] in range(0, 5), axis=1)
    in_range_of_3p_ss_i=negative_strand_introns.apply(lambda x:  x['dist_to_Annot_Start'] in range(0, 19), axis=1)

    negative_strand_introns=negative_strand_introns.assign(in_5p_ss_gencode=in_range_of_5p_ss_i)
    negative_strand_introns=negative_strand_introns.assign(in_3p_ss_gencode=in_range_of_3p_ss_i)

    return pd.concat([positive_strand_introns, positive_strand_exons, negative_strand_introns, negative_strand_exons])

def define_ss_leafcutter(sqtl_df, splicing_event):

    df=sqtl_df[sqtl_df.splicing_event==splicing_event]

    if splicing_event=='skipped_exon':
        
        range_one=[0,1,3]
        range_two=[2,4,5]
        

    elif (splicing_event=='alt_3_prime') or (splicing_event=='alt_5_prime'):

        range_one=[0,1]
        range_two=[2,3]        

   # elif (splicing_event=='two_alt_3_prime_ss') or (splicing_event=='two_alt_5_prime_ss'):

       # range_one=[0,1,2]
       # range_two=[3,4,5]  
    else:
        print('no such splicing event in this script')


    positive_strand=df[df.strand=='+']
    negative_strand=df[df.strand=='-']

    location_of_closest_exon_to_variant=positive_strand.apply(lambda x: x['variant_distance_from_splice_sites_in_cluster'].index(x['min_dis_to_splice_site_within_cluster']), axis=1)

    closest_a_to_5_prime=location_of_closest_exon_to_variant.isin(range_one)
    closest_a_to_3_prime=location_of_closest_exon_to_variant.isin(range_two)

    in_range_of_5p_ss=positive_strand.apply(lambda x:  x['min_dis_to_splice_site_within_cluster'] in range(-4, 7), axis=1)
    in_range_of_3p_ss=positive_strand.apply(lambda x:  x['min_dis_to_splice_site_within_cluster'] in range(-4, 21), axis=1)

        #for a positive strand 

    in_5p_ss_leafcutter=(in_range_of_5p_ss) & (closest_a_to_5_prime)

    in_3p_ss_leafcutter=(in_range_of_3p_ss) & (closest_a_to_3_prime)
    
    positive_strand=positive_strand.assign(in_3p_ss_leafcutter=in_3p_ss_leafcutter)
    positive_strand=positive_strand.assign(in_5p_ss_leafcutter=in_5p_ss_leafcutter)

    location_of_closest_exon_to_variant=negative_strand.apply(lambda x: x['variant_distance_from_splice_sites_in_cluster'].index(x['min_dis_to_splice_site_within_cluster']), axis=1)

    closest_a_to_5_prime=location_of_closest_exon_to_variant.isin(range_two)
    closest_a_to_3_prime=location_of_closest_exon_to_variant.isin(range_one)

    in_range_of_5p_ss=negative_strand.apply(lambda x:  x['min_dis_to_splice_site_within_cluster'] in range(-7, 4), axis=1)
    in_range_of_3p_ss=negative_strand.apply(lambda x:  x['min_dis_to_splice_site_within_cluster'] in range(-21, 4), axis=1)

    in_5p_ss_leafcutter=(in_range_of_5p_ss) & (closest_a_to_5_prime)
    in_3p_ss_leafcutter=(in_range_of_3p_ss) & (closest_a_to_3_prime)
    negative_strand=negative_strand.assign(in_3p_ss_leafcutter=in_3p_ss_leafcutter)
    negative_strand=negative_strand.assign(in_5p_ss_leafcutter=in_5p_ss_leafcutter)

    df=pd.concat([negative_strand, positive_strand], ignore_index=True)

    df['in_splice_region_leafcutter']=(df.in_3p_ss_leafcutter) | (df.in_5p_ss_leafcutter)
    
    
    
    return df